# ML2: Classify the email using the binary classification method. Email Spam
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;detection has two states: a) Normal State - Not Spam, b) Abnormal State  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- Spam. Use K-Nearest Neighbors and Support Vector Machine for  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;classification. Analyze their performance.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Dataset: https://www.kaggle.com/datasets/balaka18/email-spam-classification-dataset-csv  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(download "emails.csv" and keep it in the same folder as this file)  

## Theory

### 1. CLASSIFICATION (Unit IV)
&nbsp;&nbsp;&nbsp;Supervised learning where the target is a discrete class label.  
&nbsp;&nbsp;&nbsp;BINARY classification: 2 classes (Spam = 1 / Not spam = 0).  
&nbsp;&nbsp;&nbsp;MULTICLASS: &gt; 2 classes; handled by  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- One-vs-All (OvR): one classifier per class vs rest (k classifiers)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- One-vs-One (OvO): one classifier per pair of classes (k(k-1)/2)  
&nbsp;&nbsp;&nbsp;BALANCED vs IMBALANCED: if one class dominates, accuracy is misleading;  
&nbsp;&nbsp;&nbsp;use precision/recall/F1, stratified splits, resampling, class weights.  

### 2. DATASET
&nbsp;&nbsp;&nbsp;5172 emails; each row = an email, 3000 columns = frequency of the 3000  
&nbsp;&nbsp;&nbsp;most common words (BAG-OF-WORDS representation), last column Prediction  
&nbsp;&nbsp;&nbsp;(1 = spam, 0 = not spam). "Email No." is an ID and is dropped.  

### 3. K-NEAREST NEIGHBOURS (KNN)
&nbsp;&nbsp;&nbsp;A LAZY, NON-PARAMETRIC, instance-based learner - no explicit training;  
&nbsp;&nbsp;&nbsp;stores the data and classifies a new point by MAJORITY VOTE of its k  
&nbsp;&nbsp;&nbsp;nearest training points.  
&nbsp;&nbsp;&nbsp;Distance metrics: Euclidean sqrt(SUM (xi - yi)^2), Manhattan SUM |xi - yi|,  
&nbsp;&nbsp;&nbsp;Minkowski (SUM |xi - yi|^p)^(1/p).  
&nbsp;&nbsp;&nbsp;Choice of k: small k -&gt; noisy, overfits (high variance); large k -&gt;  
&nbsp;&nbsp;&nbsp;smooth, underfits (high bias). Odd k avoids ties in binary problems.  
&nbsp;&nbsp;&nbsp;Feature scaling is essential because distance depends on magnitude.  
&nbsp;&nbsp;&nbsp;Prediction cost O(n d) per query (n samples, d features).  

### 4. SUPPORT VECTOR MACHINE (SVM)
&nbsp;&nbsp;&nbsp;Finds the HYPERPLANE w.x + b = 0 that separates the classes with the  
&nbsp;&nbsp;&nbsp;MAXIMUM MARGIN. Margin = 2/||w||. The training points lying on the margin  
&nbsp;&nbsp;&nbsp;boundaries are the SUPPORT VECTORS.  
&nbsp;&nbsp;&nbsp;Hard margin: minimise ||w||^2/2 s.t. y\_i (w.x\_i + b) &gt;= 1.  
&nbsp;&nbsp;&nbsp;Soft margin: allows violations with slack xi\_i, penalty C:  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;minimise ||w||^2/2 + C SUM xi\_i  
&nbsp;&nbsp;&nbsp;(large C -&gt; narrow margin, fewer errors on train; small C -&gt; wider margin)  
&nbsp;&nbsp;&nbsp;KERNEL TRICK: maps data to higher dimension implicitly for non-linear  
&nbsp;&nbsp;&nbsp;boundaries. Kernels: linear, polynomial (x.y + c)^d, RBF  
&nbsp;&nbsp;&nbsp;exp(-gamma ||x - y||^2), sigmoid.  
&nbsp;&nbsp;&nbsp;Works well with high-dimensional sparse data such as text.  

### 5. EVALUATION METRICS
&nbsp;&nbsp;&nbsp;Confusion matrix:            Predicted  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Neg     Pos  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Actual Neg     TN      FP  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Actual Pos     FN      TP  
&nbsp;&nbsp;&nbsp;Accuracy  = (TP + TN) / (TP + TN + FP + FN)  
&nbsp;&nbsp;&nbsp;Precision = TP / (TP + FP)   (of predicted spam, how many are spam)  
&nbsp;&nbsp;&nbsp;Recall    = TP / (TP + FN)   (of real spam, how many were caught)  
&nbsp;&nbsp;&nbsp;F1-score  = 2 P R / (P + R)  (harmonic mean)  
&nbsp;&nbsp;&nbsp;Micro-average: pool TP/FP/FN of all classes then compute.  
&nbsp;&nbsp;&nbsp;Macro-average: compute per class then take the unweighted mean.  
&nbsp;&nbsp;&nbsp;CROSS-VALIDATION (k-fold): split data into k folds, train on k-1 and test  
&nbsp;&nbsp;&nbsp;on the remaining one, repeat k times, average -&gt; reliable estimate.  

### Conclusion
Both classifiers detect spam well; SVM (linear / RBF) usually
achieves higher accuracy and precision than KNN on high-dimensional word-count  
features, while KNN is simple but slower at prediction time.

In [ ]:
%matplotlib inline

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

DATA = "emails.csv"   # CSV file kept next to this notebook

### 1. Read & pre-process

In [ ]:
df = pd.read_csv(DATA)
print("Shape:", df.shape)
print(df.iloc[:5, :8])
print("Missing values:", df.isnull().sum().sum())
df = df.dropna()

X = df.drop(columns=["Email No.", "Prediction"], errors="ignore")
y = df["Prediction"]
print("\nClass distribution (0 = Not spam, 1 = Spam):\n", y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

### 2. KNN - choose k

In [ ]:
print("\nKNN accuracy for different k:")
k_values = range(1, 16, 2)
k_acc = []
for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, knn_k.predict(X_test_s))
    k_acc.append(acc)
    print(f"  k = {k:2d}  accuracy = {acc:.4f}")
best_k = list(k_values)[int(np.argmax(k_acc))]
print("Best k =", best_k)

plt.plot(list(k_values), k_acc, marker="o")
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.title("KNN: accuracy vs k")
plt.show()

knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_s, y_train)
y_pred_knn = knn.predict(X_test_s)

### 3. SVM

In [ ]:
svm_lin = SVC(kernel="linear", C=1.0)
svm_lin.fit(X_train_s, y_train)
y_pred_lin = svm_lin.predict(X_test_s)

svm_rbf = SVC(kernel="rbf", C=10, gamma="scale")
svm_rbf.fit(X_train_s, y_train)
y_pred_rbf = svm_rbf.predict(X_test_s)

### 4. Performance analysis

In [ ]:
def metrics(name, y_true, y_pred):
    return {"Model": name,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred)}


preds = {f"KNN (k={best_k})": y_pred_knn,
         "SVM (linear)": y_pred_lin,
         "SVM (RBF)": y_pred_rbf}

results = pd.DataFrame([metrics(n, y_test, p) for n, p in preds.items()])
print("\nPerformance comparison:\n", results.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, p) in zip(axes, preds.items()):
    print(f"\n{name}\nConfusion matrix:\n{confusion_matrix(y_test, p)}")
    print(classification_report(y_test, p,
                                target_names=["Not Spam", "Spam"]))
    sns.heatmap(confusion_matrix(y_test, p), annot=True, fmt="d",
                cmap="Blues", ax=ax,
                xticklabels=["Not Spam", "Spam"],
                yticklabels=["Not Spam", "Spam"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

# 5-fold cross-validation for a more reliable comparison
print("5-fold cross-validation accuracy:")
for name, model in [(f"KNN (k={best_k})", KNeighborsClassifier(best_k)),
                    ("SVM (linear)", SVC(kernel="linear"))]:
    scores = cross_val_score(model, scaler.transform(X), y, cv=5)
    print(f"  {name:<14} mean = {scores.mean():.4f}  std = {scores.std():.4f}")

print("\nBest model by F1:", results.loc[results["F1"].idxmax(), "Model"])